<a href="https://colab.research.google.com/github/Zeeshan-Khatib-5830521/Projects/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

PROMPT: For each table in tables, loop through and print the table name,number of rows and columns, and the first 5 records.


In [4]:
from IPython.display import display

for name, df in tables.items():
    print(f"=== Table: {name} ({df.shape[0]} rows, {df.shape[1]} columns) ===")
    display(df.head(5))
    print("\n")

=== Table: orders (14494 rows, 5 columns) ===


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00




=== Table: order_status_events (56812 rows, 4 columns) ===


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z




=== Table: order_lines (36120 rows, 5 columns) ===


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)



=== Table: restaurants (67 rows, 5 columns) ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01




=== Table: customers (4200 rows, 3 columns) ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


PROMPT: For each table in tables, display the number of missing values in each column.

In [5]:
for name, df in tables.items():
    print(f"=== Missing values in {name} ===")
    print(df.isnull().sum())
    print("\n")

=== Missing values in orders ===
order_id         0
customer_id      0
restaurant_id    0
order_ts         0
delivery_fee     0
dtype: int64


=== Missing values in order_status_events ===
event_id    0
order_id    0
status      0
event_ts    0
dtype: int64


=== Missing values in order_lines ===
order_id      0
line_no       0
item_name     0
quantity      0
unit_price    0
dtype: int64


=== Missing values in payments ===
payment_id      0
order_id        0
payment_type    0
amount          0
paid_at         0
dtype: int64


=== Missing values in restaurants ===
restaurant_id      0
restaurant_name    0
cuisine            0
city               0
effective_from     0
dtype: int64


=== Missing values in customers ===
customer_id    0
signup_date    0
home_city      0
dtype: int64




In [7]:
tables['payments']

,payment_id,order_id,payment_type,amount,paid_at
0,P014735,O013022,charge,88.49,2026-06-18 21:41:37
1,P004116,O003642,charge,52.99,2026-03-31 18:04:12
2,P009951,O008786,charge,67.49,2026-05-13 16:26:11
3,P015198,O013427,charge,83.49,2026-06-21 22:24:53
4,P011527,O010189,charge,21.99,2026-05-25 11:56:06
...,...,...,...,...,...
16418,P005481,O004848,charge,58.49,2026-04-10 17:41:20
16419,P006996,O006185,charge,38.49,2026-04-21 12:31:11
16420,P010377,O009162,charge,26.49,2026-05-16 18:35:09
16421,P002182,O001920,charge,27.49,2026-03-17 17:48:45


PROMPT: For each table in tables, check the number of completely
duplicated rows and display the results.

In [8]:
for name, df in tables.items():
    num_duplicates = df.duplicated().sum()
    print(f"=== Duplicated rows in {name} ===")
    print(f"Number of fully duplicated rows: {num_duplicates}")
    print("\n")

=== Duplicated rows in orders ===
Number of fully duplicated rows: 0


=== Duplicated rows in order_status_events ===
Number of fully duplicated rows: 0


=== Duplicated rows in order_lines ===
Number of fully duplicated rows: 0


=== Duplicated rows in payments ===
Number of fully duplicated rows: 30


=== Duplicated rows in restaurants ===
Number of fully duplicated rows: 0


=== Duplicated rows in customers ===
Number of fully duplicated rows: 0




I see that there are 30 duplicated rows in the payments table.

In [10]:
# Filter payments to show only the rows that are completely duplicated (keeping all occurrences)
payments_df = tables['payments']
duplicated_rows = payments_df[payments_df.duplicated(keep=False)]

# Sort by payment_id to see the duplicates next to each other
duplicated_rows.sort_values(by='payment_id').head(10)

,payment_id,order_id,payment_type,amount,paid_at
5949,P000219,O000185,charge,25.49,2026-03-02 19:05:43
13373,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1673,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4665,P000927,O000815,charge,23.99,2026-03-08 11:53:53
6582,P000998,O000874,charge,50.99,2026-03-08 19:19:26
3747,P000998,O000874,charge,50.99,2026-03-08 19:19:26
975,P001859,O001631,charge,38.49,2026-03-14 22:43:50
5224,P001859,O001631,charge,38.49,2026-03-14 22:43:50
15431,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9230,P002189,O001926,charge,41.99,2026-03-17 18:22:24


We can see that the same exact rows have been repeated, we will have to remove these duplicated rows at a later stage.

PROMPT: Display the data types of all columns in each table.

In [11]:
for name, df in tables.items():
    print(f"=== Data Types for {name} ===")
    print(df.dtypes)
    print("\n")

=== Data Types for orders ===
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


=== Data Types for order_status_events ===
event_id    object
order_id    object
status      object
event_ts    object
dtype: object


=== Data Types for order_lines ===
order_id       object
line_no         int64
item_name      object
quantity        int64
unit_price    float64
dtype: object


=== Data Types for payments ===
payment_id       object
order_id         object
payment_type     object
amount          float64
paid_at          object
dtype: object


=== Data Types for restaurants ===
restaurant_id      object
restaurant_name    object
cuisine            object
city               object
effective_from     object
dtype: object


=== Data Types for customers ===
customer_id    object
signup_date    object
home_city      object
dtype: object




PROMPT: For each table, identify potential primary keys and check
whether they contain duplicate values.

In [12]:
primary_key_candidates = {
    'orders': 'order_id',
    'order_status_events': 'event_id',
    'order_lines': ['order_id', 'line_no'],
    'payments': 'payment_id',
    'restaurants': 'restaurant_id',
    'customers': 'customer_id'
}

for name, pk_cols in primary_key_candidates.items():
    df = tables[name]

    # Handle composite or single keys
    if isinstance(pk_cols, list):
        total_rows = len(df)
        unique_keys = len(df.groupby(pk_cols))
        has_nulls = df[pk_cols].isnull().any().any()
    else:
        total_rows = len(df)
        unique_keys = df[pk_cols].nunique()
        has_nulls = df[pk_cols].isnull().any()

    is_unique = (total_rows == unique_keys)

    print(f"=== Table: {name} ===")
    print(f"Candidate Key: {pk_cols}")
    print(f"Total Rows: {total_rows:,} | Unique Keys: {unique_keys:,}")
    print(f"Is Unique? {is_unique} | Contains Nulls in Key? {has_nulls}")
    print("\n")

=== Table: orders ===
Candidate Key: order_id
Total Rows: 14,494 | Unique Keys: 14,494
Is Unique? True | Contains Nulls in Key? False


=== Table: order_status_events ===
Candidate Key: event_id
Total Rows: 56,812 | Unique Keys: 56,812
Is Unique? True | Contains Nulls in Key? False


=== Table: order_lines ===
Candidate Key: ['order_id', 'line_no']
Total Rows: 36,120 | Unique Keys: 36,120
Is Unique? True | Contains Nulls in Key? False


=== Table: payments ===
Candidate Key: payment_id
Total Rows: 16,423 | Unique Keys: 16,393
Is Unique? False | Contains Nulls in Key? False


=== Table: restaurants ===
Candidate Key: restaurant_id
Total Rows: 67 | Unique Keys: 64
Is Unique? False | Contains Nulls in Key? False


=== Table: customers ===
Candidate Key: customer_id
Total Rows: 4,200 | Unique Keys: 4,200
Is Unique? True | Contains Nulls in Key? False




Looking at the above results, I decide that

The date or time related columns must be converted into pandat datetime object data type. I will do this in section 3.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

PROMPT: Dont give any output in any cell, just tell me which tables wil i require and which keys will I have to join on in order to get the final table which is: One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes.

In the final table Each row will represent the net revenue and average delivery time in a specific week, in a specific city and cuisine.

Inorder to form this final table I will require the orders, restaurants, order_status_events & payments table.

orders will join with restaurants on the restaurant_id, orders will join to order_status_events on the order_id, orders will join with payments on order_id.




## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

PROMPT: Create a copy of each original DataFrame for cleaning.
Keep the original tables unchanged.

In [13]:
# Create copies of each original DataFrame to keep the raw data intact
orders_clean = tables['orders'].copy()
order_status_events_clean = tables['order_status_events'].copy()
order_lines_clean = tables['order_lines'].copy()
payments_clean = tables['payments'].copy()
restaurants_clean = tables['restaurants'].copy()
customers_clean = tables['customers'].copy()

print("Copies created successfully:")
print(f"- orders_clean: {orders_clean.shape}")
print(f"- order_status_events_clean: {order_status_events_clean.shape}")
print(f"- order_lines_clean: {order_lines_clean.shape}")
print(f"- payments_clean: {payments_clean.shape}")
print(f"- restaurants_clean: {restaurants_clean.shape}")
print(f"- customers_clean: {customers_clean.shape}")

Copies created successfully:
- orders_clean: (14494, 5)
- order_status_events_clean: (56812, 4)
- order_lines_clean: (36120, 5)
- payments_clean: (16423, 5)
- restaurants_clean: (67, 5)
- customers_clean: (4200, 3)


Add text here as you go along. Show that you are staying in control of the build!

Rows in payments_clean before dropping duplicates: 16393
Rows in payments_clean after dropping duplicates: 16393
Successfully removed 0 duplicate rows.


In [19]:
orders_clean.duplicated().sum()

np.int64(0)

PROMPT: convert the data type in all the date or time or timestamp related columns in all the tables into datetime object

In [20]:
# Convert date and timestamp columns to datetime objects in clean DataFrames
orders_clean['order_ts'] = pd.to_datetime(orders_clean['order_ts'])
order_status_events_clean['event_ts'] = pd.to_datetime(order_status_events_clean['event_ts'])
payments_clean['paid_at'] = pd.to_datetime(payments_clean['paid_at'])
restaurants_clean['effective_from'] = pd.to_datetime(restaurants_clean['effective_from'])
customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])

# Verification of the conversions
print("Verifying updated column data types:")
print(f"- orders_clean['order_ts']: {orders_clean['order_ts'].dtype}")
print(f"- order_status_events_clean['event_ts']: {order_status_events_clean['event_ts'].dtype}")
print(f"- payments_clean['paid_at']: {payments_clean['paid_at'].dtype}")
print(f"- restaurants_clean['effective_from']: {restaurants_clean['effective_from'].dtype}")
print(f"- customers_clean['signup_date']: {customers_clean['signup_date'].dtype}")

Verifying updated column data types:
- orders_clean['order_ts']: datetime64[ns, UTC]
- order_status_events_clean['event_ts']: datetime64[ns, UTC]
- payments_clean['paid_at']: datetime64[ns]
- restaurants_clean['effective_from']: datetime64[ns]
- customers_clean['signup_date']: datetime64[ns]


In [25]:
type(payments_clean['paid_at'][0])
type(restaurants_clean['effective_from'][0])

pandas._libs.tslibs.timestamps.Timestamp

In [26]:
payments_clean['paid_at']

,paid_at
0,2026-06-18 21:41:37
1,2026-03-31 18:04:12
2,2026-05-13 16:26:11
3,2026-06-21 22:24:53
4,2026-05-25 11:56:06
...,...
16418,2026-04-10 17:41:20
16419,2026-04-21 12:31:11
16420,2026-05-16 18:35:09
16421,2026-03-17 17:48:45


PROMPT: Identify inconsistent category names and standardise values that represent the same category.

In [27]:
# Inspect unique categorical values before standardization
print("=== Unique cuisines in restaurants_clean ===")
print(restaurants_clean['cuisine'].unique())

print("\n=== Unique cities in restaurants_clean ===")
print(restaurants_clean['city'].unique())

print("\n=== Unique home_cities in customers_clean ===")
print(customers_clean['home_city'].unique())

print("\n=== Unique payment_types in payments_clean ===")
print(payments_clean['payment_type'].unique())

# Standardize text columns (strip whitespace, ensure uniform title casing)
restaurants_clean['cuisine'] = restaurants_clean['cuisine'].str.strip().str.title()
restaurants_clean['city'] = restaurants_clean['city'].str.strip().str.title()
customers_clean['home_city'] = customers_clean['home_city'].str.strip().str.title()
payments_clean['payment_type'] = payments_clean['payment_type'].str.strip().str.lower()

print("\n=== After Standardization ===")
print("Unique Cuisines:", restaurants_clean['cuisine'].unique())
print("Unique Restaurant Cities:", restaurants_clean['city'].unique())
print("Unique Customer Cities:", customers_clean['home_city'].unique())
print("Unique Payment Types:", payments_clean['payment_type'].unique())

=== Unique cuisines in restaurants_clean ===
['Vegan' 'Burgers' 'Japanese' 'Indian' 'Thai' 'Italian' 'mexican'
 'Chinese' 'japanese' 'thai' 'Burgers ' 'Mexican' 'Japanese ' 'Indian '
 'Italian ']

=== Unique cities in restaurants_clean ===
['London' 'Bristol' 'Birmingham' 'LEEDS' 'Manchester' 'MANCHESTER' 'Leeds'
 'LONDON']

=== Unique home_cities in customers_clean ===
['London' 'Leeds' 'Manchester' 'Bristol' 'Birmingham']

=== Unique payment_types in payments_clean ===
['charge' 'refund']

=== After Standardization ===
Unique Cuisines: ['Vegan' 'Burgers' 'Japanese' 'Indian' 'Thai' 'Italian' 'Mexican'
 'Chinese']
Unique Restaurant Cities: ['London' 'Bristol' 'Birmingham' 'Leeds' 'Manchester']
Unique Customer Cities: ['London' 'Leeds' 'Manchester' 'Bristol' 'Birmingham']
Unique Payment Types: ['charge' 'refund']


In [29]:
restaurants_clean['city'].unique()

array(['London', 'Bristol', 'Birmingham', 'Leeds', 'Manchester'],
      dtype=object)

PROMPT: Remove leading and trailing spaces and standardise
capitalisation in the relevant text columns.

In [30]:
# Remove leading/trailing spaces and standardize capitalization on additional text columns
restaurants_clean['restaurant_name'] = restaurants_clean['restaurant_name'].str.strip().str.title()
order_lines_clean['item_name'] = order_lines_clean['item_name'].str.strip().str.title()

# Quick verification
print("=== Sample of cleaned restaurant names ===")
print(restaurants_clean['restaurant_name'].head())
print("\n=== Sample of cleaned item names ===")
print(order_lines_clean['item_name'].unique()[:10])

=== Sample of cleaned restaurant names ===
0      Fresh Vegan 64
1    Royal Burgers 12
2    Grand Burgers 28
3    Royal Japanese 7
4    Corner Indian 26
Name: restaurant_name, dtype: object

=== Sample of cleaned item names ===
['Ramen' 'Soft Drink' 'Side Salad' 'Burrito' 'Fries' 'Fried Rice'
 'Chicken Tikka' 'Pad Thai' 'Dessert' 'Falafel Wrap']


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.